# Main-text Figure 2 (panels A–D)

This notebook reads the compact public model-summary table. Panels E–H are the manually checked PowerPoint SEM diagrams used in the manuscript.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import ultraplot as uplt

DATA = Path("../data/main_figures/figure2")
OUTPUT = Path("../outputs/main_figures")
OUTPUT.mkdir(parents=True, exist_ok=True)

models = pd.read_csv(DATA / "model_compare.csv")


In [ ]:
TAXA = ["Mammals", "Birds", "Amphibians", "Reptiles"]
WINDOWS = ["r1", "r3", "r6", "r12"]
WINDOW_LABELS = ["1 mo", "3 mo", "6 mo", "12 mo"]
MODEL_LADDER = {
    "M0": "M_NPPact_only",
    "M1": "M_NPPact_HANPPharv",
    "M2": "M_NPPact_HANPPharv_Dfreq",
    "M3": "M_NPPact_HANPPharv_Dfreq_Dmag",
}
STEP_PAIRS = [
    ("M1 - M0", "M1", "M0"),
    ("M2 - M1", "M2", "M1"),
    ("M3 - M2", "M3", "M2"),
]
STEP_LABELS = {
    "M1 - M0": r"$+\mu_{H}$",
    "M2 - M1": r"$+\mu_{\mathrm{D}_{freq}}$",
    "M3 - M2": r"$+\mu_{\mathrm{D}_{mag}}$",
}


def step_delta(table, step, new_key, old_key):
    columns = ["Taxon", "Window", "Adj_R2_Richness", "AIC_Richness"]
    new = table.loc[table["Model"] == MODEL_LADDER[new_key], columns]
    old = table.loc[table["Model"] == MODEL_LADDER[old_key], columns]
    merged = new.merge(old, on=["Taxon", "Window"], suffixes=("_new", "_old"))
    merged["Step"] = step
    merged["Delta_Adj_R2"] = merged["Adj_R2_Richness_new"] - merged["Adj_R2_Richness_old"]
    merged["Delta_AIC"] = merged["AIC_Richness_new"] - merged["AIC_Richness_old"]
    return merged


def matrix(table, value):
    return (
        table.pivot(index="Taxon", columns="Window", values=value)
        .reindex(index=TAXA, columns=WINDOWS)
    )


def annotate_matrix(ax, values, fmt, fontsize=7.2):
    for row_index, row in enumerate(values.index):
        for column_index, column in enumerate(values.columns):
            value = values.loc[row, column]
            text = "" if pd.isna(value) else fmt.format(value)
            if text and float(text) == 0:  # no sign on a value that rounds to zero
                text = text.lstrip("+-")
            ax.text(
                column_index, row_index, text,
                ha="center", va="center", fontsize=fontsize, color="black"
            )


delta_df = pd.concat(
    [step_delta(models, *step_spec) for step_spec in STEP_PAIRS],
    ignore_index=True,
)


In [ ]:
uplt.rc.update({
    "font.size": 8,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "tick.labelsize": 8,
})

lim_adj = np.nanmax(np.abs(delta_df["Delta_Adj_R2"].to_numpy()))
lim_aic = np.nanmax(np.abs(delta_df["Delta_AIC"].to_numpy()))

fig, axs = uplt.subplots(
    nrows=1,
    ncols=4,
    refwidth=2.05,
    refheight=2.15,
    wspace=0.85,
    share=False,
    right=7,
)

step_h = STEP_PAIRS[0][0]
step_dfreq = STEP_PAIRS[1][0]
panel_specs = [
    ("A", step_h, "r2"),
    ("B", step_h, "aic"),
    ("C", step_dfreq, "r2"),
    ("D", step_dfreq, "aic"),
]

m_r2 = None
m_aic = None
for index, (panel, step_name, metric) in enumerate(panel_specs):
    ax = axs[index]
    subset = delta_df.loc[delta_df["Step"] == step_name]
    if metric == "r2":
        values = matrix(subset, "Delta_Adj_R2")
        mappable = ax.imshow(
            values.to_numpy(), cmap="PiYG", vmin=-lim_adj, vmax=lim_adj
        )
        annotate_matrix(ax, values, fmt="{:+.3f}")
        metric_title = r"$\Delta$ adj. $R^2$"
        m_r2 = mappable
    else:
        values = matrix(subset, "Delta_AIC")
        mappable = ax.imshow(
            values.to_numpy(), cmap="RdBu_r", vmin=-lim_aic, vmax=lim_aic
        )
        annotate_matrix(ax, values, fmt="{:+.1f}")
        metric_title = r"$\Delta$ AIC"
        m_aic = mappable

    ax.format(
        title=rf"{panel}  {metric_title}  {STEP_LABELS[step_name]}",
        xticks=np.arange(len(WINDOWS)),
        xticklabels=WINDOW_LABELS,
        yticks=np.arange(len(TAXA)),
        yticklabels=TAXA if index == 0 else [],
        xlabel="Window",
    )

axs[0].format(ylabel="Taxon")
cax_r2 = fig.add_axes([0.93, 0.53, 0.012, 0.30])
cax_aic = fig.add_axes([0.93, 0.15, 0.012, 0.30])
fig.colorbar(m_r2, cax=cax_r2, label=r"$\Delta$ adj. $R^2$ (absolute)")
fig.colorbar(m_aic, cax=cax_aic, label=r"$\Delta$ AIC")

fig.savefig(OUTPUT / "Fig2_AD.png", dpi=300, bbox_inches=None, facecolor="white")
fig
